In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

import joblib
import os

# Display Markdown
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Libraries Loaded!</span>"))

In [ ]:
data_path    = r"D:/Education/AiQuest/DS,ML,DL/Projects/Car Price Prediction/data/car data.csv"
data= pd.read_csv(data_path)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Data Loaded</span>"))

In [ ]:
display(Markdown(f"**Rows:** {data.shape[0]}, **Columns:** {data.shape[1]}"))
display(data.head(3))

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Feature Engineering!</h2>

In [ ]:
display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Creating Car_Age Feature</span>"))

# Car_Age = number of years since the car was manufactured, relative to the most recent model year present in the dataset
data['Car_Age'] = data['Year'].max() + 1 - data['Year']

display(data[['Year', 'Car_Age']].head())

In [ ]:
X = data.drop(columns=['Selling_Price', 'Car_Name', 'Year'])  # features only, raw categoricals still intact
y = data['Selling_Price']

In [ ]:
x_data = data.drop(['Car_Name','Year'], axis=1)

display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Dataset after modification</span>"))

display(x_data.head())

In [ ]:
cols_to_encode = ['Fuel_Type', 'Seller_Type', 'Transmission']
data_encoded = pd.get_dummies(x_data, columns=cols_to_encode, drop_first=True,dtype=int)
display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Dataset after One Hot Encoding</span>"))
display(data_encoded.head(3))

Fuel_Type, Seller_Type, and Transmission are all nominal categories — there's no natural order between "Diesel" and "Petrol", or between "Manual" and "Automatic". Label encoding would map these to arbitrary integers (e.g. CNG=0, Diesel=1, Petrol=2), which introduces a fake ordinal relationship ("Petrol > Diesel > CNG") that linear or distance-based models (Linear Regression, KNN, SVM) could mistakenly learn from.

One-hot encoding avoids this by representing each category as its own independent binary column, with no implied ranking. It's a safe default here because cardinality is very low:

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Load data and Split!</h2>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
display(Markdown(f"""
<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_train :</b>
<span style='color:black; font-weight:bold;'>{X_train.shape}</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>X_test  :</b>
<span style='color:black; font-weight:bold;'>{X_test.shape}</span><br>
<span style='font-size:16px;'>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_train :</b>
<span style='color:black; font-weight:bold;'>{y_train.shape}</span><br>
<b style='color:#27AE60; font-weight:bold; text-shadow:0 0 3px rgba(15,52,96,0.6);'>y_test  :</b>
<span style='color:black; font-weight:bold;'>{y_test.shape}
</span>
"""))

In [ ]:
def get_outlier_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

# Kms_Driven is a feature -> bounds from X_train
kms_lower, kms_upper = get_outlier_bounds(X_train['Kms_Driven'])
X_train['Kms_Driven'] = X_train['Kms_Driven'].clip(kms_lower, kms_upper)
X_test['Kms_Driven']  = X_test['Kms_Driven'].clip(kms_lower, kms_upper)   # same bounds, no new fitting

# Selling_Price is the target -> bounds from y_train
price_lower, price_upper = get_outlier_bounds(y_train)
y_train = y_train.clip(price_lower, price_upper)
y_test  = y_test.clip(price_lower, price_upper)

In [ ]:
display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Dataset after Removing Outliers</span>"))
display(X_train.head(3))

In [ ]:
display(Markdown("## <span style='color:#27AE60; font-weight:bold; font-size:24px; font-size:24px;text-shadow:0 0 5px rgba(15,52,96,0.6);'>Separate Categorical & Numerical Columns</span>"))

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Categorical Columns</span>"))
display(categorical_cols)

display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Numerical Columns</span>"))
display(numerical_cols)

In [ ]:
preprocessor_scaled = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
    ('num', StandardScaler(), numerical_cols)
], remainder='passthrough')

# --- Preprocessor for tree-based models (Random Forest, Decision Tree, XGBoost) ---
preprocessor_tree = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
], remainder='passthrough')

display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Two Preprocessors Created</span>"))

 # <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Training Models!</h2>

In [ ]:
%pip install lightgbm --quiet

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [ ]:
pipelines = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('model', LinearRegression())
    ]),
    'Ridge α=0.1': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('model', Ridge(alpha=0.1, random_state=42))
    ]),
    'Ridge α=1': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('model', Ridge(alpha=1, random_state=42))
    ]),
    'Ridge α=10': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('model', Ridge(alpha=10, random_state=42))
    ]),
    'Random Forest n=100': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    'Random Forest n=200': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', RandomForestRegressor(n_estimators=200, random_state=42))
    ]),
    'XGBoost (default)': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', XGBRegressor(random_state=42))
    ]),
    'XGBoost (tuned)': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42))
    ]),
    'LightGBM (default)': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', LGBMRegressor(random_state=42, verbose=-1))
    ]),
    'LightGBM (tuned)': Pipeline([
        ('preprocessor', preprocessor_tree),
        ('model', LGBMRegressor(n_estimators=300, num_leaves=15, learning_rate=0.05, random_state=42, verbose=-1))
    ])
}

In [ ]:
results = []

for name, pipeline in pipelines.items():

    # Train
    pipeline.fit(X_train, y_train)

    # Predictions
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))  # sqrt() used instead of squared=False for compatibility across sklearn versions

    results.append(
        [
            name,
            train_r2,
            test_r2,
            test_rmse
        ]
    )

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Models are Trained</span>"))

In [ ]:
results

 # <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparison Table</h2>

In [ ]:
comparison_table = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train R²",
        "Test R²",
        "Test RMSE"
    ]
)

comparison_table["Train R²"] = comparison_table["Train R²"].round(4)
comparison_table["Test R²"] = comparison_table["Test R²"].round(4)
comparison_table["Test RMSE"] = comparison_table["Test RMSE"].round(4)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Created</span>"))

In [ ]:
print("\nModel Comparison Table")
print("=" * 80)

print(comparison_table)

In [ ]:
sorted_comparison_table = comparison_table.sort_values(
    by='Test R²',
    ascending=False
).reset_index(drop=True)

display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Comparision Table Sorted on Test R² </span>"))

In [ ]:
print("\nSorted Model Comparison Table")
print("=" * 80)

print(sorted_comparison_table)

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Best Model</h2>

In [ ]:
best_model = sorted_comparison_table.iloc[0]

print("\nBest Model")
print("=" * 80)

print(f"Model     : {best_model['Model']}")
print(f"Test R²   : {best_model['Test R²']}")
print(f"Test RMSE : {best_model['Test RMSE']:.4f}")

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Model Selection</h2>

In [ ]:
best_row = sorted_comparison_table.iloc[0]
best_name, best_train_r2, best_test_r2, best_rmse = best_row['Model'], best_row['Train R²'], best_row['Test R²'], best_row['Test RMSE']
gap = best_train_r2 - best_test_r2

lin_r2 = comparison_table.loc[comparison_table['Model'] == 'Linear Regression', 'Test R²'].values[0]
ridge_rows = comparison_table[comparison_table['Model'].str.startswith('Ridge')]
tree_rows = comparison_table[comparison_table['Model'].str.contains('Random Forest|XGBoost|LightGBM')]

overfit_note = (
    "generalizes well -- Train and Test R² are close, so the model isn't just memorizing the training data"
    if gap < 0.05 else
    "shows some degree of overfitting -- Train R² is noticeably higher than Test R², meaning it fits the training data more closely than it generalizes to unseen data"
)

is_tree_based = any(k in best_name for k in ['Random Forest', 'XGBoost', 'LightGBM'])
eda_note = (
    "This lines up with the EDA: Present_Price showed the strongest correlation with Selling_Price, and Car_Age showed a clear negative relationship (older cars sell for less) -- a linear model can capture both of these relationships directly, without needing the extra complexity of a tree ensemble."
    if not is_tree_based else
    "This suggests the EDA's linear correlations (Present_Price positively, Car_Age negatively) don't tell the whole story -- there are likely feature interactions or non-linear effects (e.g. how Kms_Driven interacts with Car_Age, or how Fuel_Type shifts price differently across price brackets) that only a tree-based model can capture."
)

display(Markdown(f"""
### Why `{best_name}` performs best on this dataset

- **Test R²:** {best_test_r2:.4f} | **Train R²:** {best_train_r2:.4f} | **Test RMSE:** {best_rmse:.4f} (in the same units as `Selling_Price`, i.e. lakhs)
- It was the top performer among all {len(comparison_table)} model configurations tried (Linear Regression, 3 Ridge alphas, 2 Random Forest sizes, default + tuned XGBoost, default + tuned LightGBM).

### Overfitting check (Train R² vs Test R²)

The model {overfit_note} (Train−Test gap = {gap:.4f}).

### Connection to EDA findings

{eda_note}
"""))

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Feature Importance</h2>

In [ ]:
best_pipeline = pipelines[best_name]
best_estimator = best_pipeline.named_steps['model']

if hasattr(best_estimator, 'feature_importances_'):
    display(Markdown("### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Feature Importances (best model is tree-based)</span>"))

    feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
    importances = best_estimator.feature_importances_

    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False).reset_index(drop=True)

    display(importance_df)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
    plt.title(f'Feature Importance – {best_name}')
    plt.tight_layout()
    plt.show()

    top_feature = importance_df.iloc[0]['Feature']
    display(Markdown(f"**Top driver of price:** `{top_feature}`. Compare this against the EDA correlation heatmap -- if `Present_Price`/`Car_Age`-derived features dominate here too, it confirms the EDA's correlation-based findings; if a different feature (e.g. a `Fuel_Type` or `Transmission` dummy) ranks unexpectedly high, that's a non-linear/interaction effect the correlation heatmap alone couldn't reveal."))
else:
    display(Markdown(f"### <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Feature Importance</span>\n\n`{best_name}` is not tree-based, so it has no `feature_importances_` attribute. For a linear model like this, relative feature influence can instead be read from `best_estimator.coef_` (on the standardized features), which plays the same role as EDA's `Present_Price`/`Car_Age` correlations already suggested."))

# <h2 style= 'text-align:center; color:#27AE60; font-weight:bold;font-size:40px;  text-shadow:0 0 3px rgba(15,52,96,0.6);'>Save Best Model Pipeline</h2>

In [ ]:
final_pipeline = pipelines[best_model['Model']]
display(Markdown(f"### <span style='color:#27AE60; font-weight:bold;font-size:20px;'>Using already-trained pipeline: {best_model['Model']}</span>"))

In [ ]:
final_pipeline.fit(X_train, y_train)

In [ ]:
model_path = r'D:\Education\AiQuest\DS,ML,DL\Projects\Car Price Prediction\models'

os.makedirs(model_path, exist_ok=True)

joblib.dump(
    final_pipeline,
    os.path.join(model_path, 'best_model.pkl')
)
display(Markdown("## <span style='color:#27AE60; font-weight:bold;font-size:24px; text-shadow:0 0 3px rgba(15,52,96,0.6);'>Model saved successfully </span>"))